In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install sentence-transformers umap-learn plotly -q

In [ ]:
import numpy as np
import plotly.graph_objects as go
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("LaBSE")

LANGUAGE_SPEAKER_COUNTS = {
    "en": 1_500_000_000, "zh": 1_100_000_000, "es": 560_000_000,
    "hi": 600_000_000,   "ar": 310_000_000,   "pt": 260_000_000,
    "fr": 280_000_000,   "de": 130_000_000,   "ja": 125_000_000,
    "ko": 80_000_000,    "ru": 260_000_000,   "it": 85_000_000,
}

def concept_centroid_report(concept_word, translations: dict, n_neighbors=5):
    """
    translations: {lang_code: word_in_that_language}
    Returns centroid, per-language distance, nearest neighbors per language.
    This is a proto-dictionary-entry for the universal language.
    """
    langs  = list(translations.keys())
    words  = list(translations.values())
    embs   = model.encode(words, normalize_embeddings=True)
    
    # Speaker-count-weighted centroid (Zipfian: more speakers = more weight)
    weights = np.array([LANGUAGE_SPEAKER_COUNTS.get(l, 50_000_000) for l in langs], dtype=float)
    weights /= weights.sum()
    centroid = (embs * weights[:, None]).sum(axis=0)
    centroid /= np.linalg.norm(centroid)
    
    # Per-language distance to centroid
    dists = {l: float(1 - cosine_similarity(embs[i:i+1], centroid.reshape(1,-1))[0][0])
             for i, l in enumerate(langs)}
    
    # Nearest neighbor from each language to the centroid
    # (what concept does the centroid ACTUALLY represent in each language?)
    # For real use: embed a large vocabulary per language and find closest word
    # Here we just show the distance structure
    
    print(f"\n{'='*55}")
    print(f"Concept: '{concept_word}'")
    print(f"Weighted centroid computed from {len(langs)} languages")
    print(f"\nPer-language distance to universal centroid:")
    for lang, dist in sorted(dists.items(), key=lambda x: x[1]):
        bar = "█" * int((1 - dist) * 30)
        print(f"  {lang:4s} | {bar:<30} | dist={dist:.4f}  ({translations[lang]})")
    
    most_central  = min(dists, key=dists.get)
    most_isolated = max(dists, key=dists.get)
    print(f"\n  Most central language: {most_central} ('{translations[most_central]}')")
    print(f"  Most isolated language: {most_isolated} ('{translations[most_isolated]}')")
    print(f"  → '{translations[most_isolated]}' in {most_isolated} diverges most from the universal concept.")
    
    return centroid, dists

# ── Demo ──────────────────────────────────────────────────────────────────────
centroid_saudade, dists_saudade = concept_centroid_report("saudade", {
    "pt": "saudade", "en": "longing", "es": "añoranza",
    "de": "Sehnsucht", "fr": "mélancolie", "ja": "物悲しさ",
    "ar": "حنين", "ru": "тоска", "it": "nostalgia",
})

centroid_hygge, dists_hygge = concept_centroid_report("hygge", {
    "da": "hygge", "en": "coziness", "de": "Gemütlichkeit",
    "fr": "convivialité", "es": "acogedor", "pt": "aconchego",
    "ja": "居心地の良さ", "ru": "уют", "it": "accoglienza",
})

concept_centroid_report("justice", {
    "en": "justice", "es": "justicia", "fr": "justice",
    "de": "Gerechtigkeit", "ar": "عدالة", "zh": "正义",
    "ja": "正義", "pt": "justiça", "ru": "справедливость",
})